# PCA-Based Relative Value Backtest

Systematic RV strategy on the USD SOFR swap curve using rolling PCA residuals,
OLS regression signals, traffic-light regime filtering, and dual MTM modes.

**References:**
- J.P. Morgan "RV on the EUR swap yield curve" (Apr 2021)
- Standard Chartered PCA residual heatmap approach

## Section 1: Imports & Configuration

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytz
from IPython.display import display

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder
from TB.IRSwapsTB import IRSwapsTB
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue
from Query.IRSwaps.IRSwapStructure import IRSwapStructure

from BT.signals.pca_rv_engine import PCARVConfig, rolling_pca, pca_fly_weights, adf_test, ou_half_life
from BT.signals.regression_rv import RegressionRVConfig, rolling_regression
from BT.signals.regime_filter import RegimeFilterConfig, traffic_light
from BT.signals.rv_backtest import RVBacktestConfig, run_rv_backtest, Trade

In [ ]:
config = {
    # --- Data ---
    "data_start": "2020-01-01",
    "data_end": "2026-03-21",
    "tenors_spot": ["2Y", "3Y", "5Y", "7Y", "10Y", "15Y", "20Y", "30Y"],
    "forward_tails": [],  # e.g. ["1Y", "2Y"] for forward-starting

    # --- Fly definitions ---
    "fly_categories": {
        "standard_spot": [
            ("2Y", "5Y", "10Y"),
            ("3Y", "5Y", "7Y"),
            ("2Y", "5Y", "7Y"),
            ("5Y", "10Y", "30Y"),
            ("3Y", "7Y", "15Y"),
            ("2Y", "3Y", "5Y"),
            ("5Y", "7Y", "10Y"),
            ("7Y", "10Y", "15Y"),
            ("10Y", "15Y", "20Y"),
            ("10Y", "20Y", "30Y"),
        ],
    },
    "enabled_categories": ["standard_spot"],

    # --- PCA / Signal ---
    "weighting_method": "regression",  # "pca" or "regression"
    "pca_window_days": 130,
    "pca_input": "levels",
    "n_components": 3,
    "pca_scope": "full_curve",
    "zscore_lookback_days": 130,

    # --- Regime filter ---
    "regime_filter_enabled": True,
    "regime_reference_fly": ("2Y", "5Y", "10Y"),
    "beta_vol_window_days": 65,
    "beta_vol_zscore_window_days": 130,
    "regime_threshold": 3.0,

    # --- Entry triggers ---
    "entry_min_rsq": 0.60,
    "entry_min_residual_bp": 4.0,
    "entry_min_zscore": 1.5,

    # --- Exit triggers ---
    "exit_mean_reversion": True,
    "exit_stop_loss_sd": 2.0,
    "exit_max_holding_days": 22,
    "exit_carry_adjusted": False,

    # --- Portfolio rules ---
    "max_concurrent_trades": None,
    "no_duplicate_flies": True,

    # --- Costs ---
    "round_trip_cost_bp": 0.5,
    "min_profit_to_cost_ratio": 2.0,

    # --- MTM mode ---
    "mtm_mode": "approximate",  # "approximate" or "curve"
}
print("Config loaded")

## Section 2: Data Loading

In [ ]:
def load_rate_timeseries(config, curve_mdp, ts_builder):
    """Load historical swap rates for all tenors defined in config."""
    tz = pytz.timezone("America/New_York")
    start = datetime.datetime.strptime(config["data_start"], "%Y-%m-%d").replace(hour=17, tzinfo=tz)
    end = datetime.datetime.strptime(config["data_end"], "%Y-%m-%d").replace(hour=17, tzinfo=tz)

    # Spot tenors
    queries = [
        IRSwapQuery(curve="USD-SOFR-1D", tenor=t, value=IRSwapValue.RATE)
        for t in config["tenors_spot"]
    ]

    # Forward-starting tenors for each tail
    for tail in config["forward_tails"]:
        for spot_t in config["tenors_spot"]:
            fwd_tenor = f"{spot_t}x{tail}" if "x" not in spot_t else spot_t
            queries.append(
                IRSwapQuery(curve="USD-SOFR-1D", tenor=fwd_tenor, value=IRSwapValue.RATE)
            )

    # Build enabled fly tenors and ensure they're in the query set
    fly_tenors = set()
    for cat in config["enabled_categories"]:
        for fly in config["fly_categories"].get(cat, []):
            for t in fly:
                fly_tenors.add(t)

    for t in fly_tenors:
        if not any(q.tenor == t for q in queries):
            queries.append(
                IRSwapQuery(curve="USD-SOFR-1D", tenor=t, value=IRSwapValue.RATE)
            )

    router = {"IRS": IRSwapsTB(curve_mdp, show_tqdm=True)}
    df = ts_builder.get_timeseries(start=start, end=end, queries=queries, n_jobs=12, routers=router)
    return df

In [ ]:
curve_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
ts_builder = TimeseriesBuilder()
rates_df = load_rate_timeseries(config, curve_mdp, ts_builder)
print(f"Loaded: {rates_df.shape[0]} dates x {rates_df.shape[1]} tenors")
rates_df.tail()

## Section 3: Signal Generation

In [ ]:
def build_fly_rate_dfs(rates_df, config):
    """Extract [dates x 3] DataFrames for each fly from the full rate matrix."""
    fly_dfs = {}

    # Helper to find the right column for a tenor
    def find_col(tenor):
        for c in rates_df.columns:
            if tenor in c:
                return c
        return None

    for cat in config["enabled_categories"]:
        for fly_tuple in config["fly_categories"].get(cat, []):
            left, belly, right = fly_tuple
            c_left = find_col(left)
            c_belly = find_col(belly)
            c_right = find_col(right)
            if c_left and c_belly and c_right:
                fly_id = f"{left}/{belly}/{right}"
                fly_dfs[fly_id] = rates_df[[c_left, c_belly, c_right]].rename(
                    columns={c_left: "left", c_belly: "belly", c_right: "right"}
                ).dropna()
    return fly_dfs

fly_dfs = build_fly_rate_dfs(rates_df, config)
print(f"Built {len(fly_dfs)} fly DataFrames")

In [ ]:
pca_config = PCARVConfig(
    pca_window_days=config["pca_window_days"],
    pca_input=config["pca_input"],
    n_components=config["n_components"],
    use_correlation=False,
    zscore_lookback_days=config["zscore_lookback_days"],
)
reg_config = RegressionRVConfig(
    window_days=config["pca_window_days"],
    zscore_lookback_days=config["zscore_lookback_days"],
)

all_residuals = {}
all_zscores = {}
all_rsq = {}
all_weights = {}
fly_categories_map = {}

for cat in config["enabled_categories"]:
    for fly_tuple in config["fly_categories"].get(cat, []):
        fly_id = "/".join(fly_tuple)
        if fly_id not in fly_dfs:
            continue

        df3 = fly_dfs[fly_id]

        if config["weighting_method"] == "pca":
            # PCA on the 3 fly tenors
            result = rolling_pca(df3, pca_config)
            # Use the belly residual as the signal
            all_residuals[fly_id] = result.residuals["belly"]
            all_zscores[fly_id] = result.zscores["belly"]
            # Variance explained as proxy R-squared
            all_rsq[fly_id] = result.variance_explained.sum(axis=1)
            all_weights[fly_id] = pca_fly_weights(df3, pca_config)
        else:
            # Regression method
            fly_50_50 = 0.5 * df3["left"] + 0.5 * df3["right"] - df3["belly"]
            body = df3["belly"]
            wing_curve = df3["right"] - df3["left"]
            reg_result = rolling_regression(fly_50_50, body, wing_curve, reg_config)
            all_residuals[fly_id] = reg_result.residuals
            all_zscores[fly_id] = reg_result.zscores
            all_rsq[fly_id] = reg_result.rsq
            # Convert hedge ratios to 3-column weights
            all_weights[fly_id] = pd.DataFrame({
                "left": reg_result.hedge_ratios["left_weight"],
                "belly": 1.0,
                "right": reg_result.hedge_ratios["right_weight"],
            }, index=df3.index)

        fly_categories_map[fly_id] = cat

print(f"Computed signals for {len(all_residuals)} flies")

## Section 4: Regime Filter (Traffic Light)

In [ ]:
# Use reference fly for traffic light
ref_fly = "/".join(config["regime_reference_fly"])
if ref_fly in fly_dfs and config["regime_filter_enabled"]:
    ref_df = fly_dfs[ref_fly]
    ref_fly_50_50 = 0.5 * ref_df["left"] + 0.5 * ref_df["right"] - ref_df["belly"]
    ref_body = ref_df["belly"]
    ref_curve = ref_df["right"] - ref_df["left"]
    ref_reg = rolling_regression(ref_fly_50_50, ref_body, ref_curve, reg_config)

    tl_config = RegimeFilterConfig(
        beta_vol_window_days=config["beta_vol_window_days"],
        beta_vol_zscore_window_days=config["beta_vol_zscore_window_days"],
        threshold=config["regime_threshold"],
    )
    tl_result = traffic_light(ref_reg.betas_body, ref_reg.betas_curve, tl_config)
    regime_series = tl_result["regime"]
    print(f"Traffic light: {(regime_series == 'red').sum()} red days, {(regime_series == 'green').sum()} green days")
else:
    regime_series = pd.Series("green", index=rates_df.index)
    print("Regime filter disabled — all green")

## Section 5: Backtest Execution

In [ ]:
bt_config = RVBacktestConfig(
    mtm_mode=config.get("mtm_mode", "approximate"),
    entry_min_rsq=config["entry_min_rsq"],
    entry_min_residual_bp=config["entry_min_residual_bp"],
    entry_min_zscore=config["entry_min_zscore"],
    exit_mean_reversion=config["exit_mean_reversion"],
    exit_stop_loss_sd=config["exit_stop_loss_sd"],
    exit_max_holding_days=config["exit_max_holding_days"],
    exit_carry_adjusted=config["exit_carry_adjusted"],
    max_concurrent_trades=config["max_concurrent_trades"],
    no_duplicate_flies=config["no_duplicate_flies"],
    round_trip_cost_bp=config["round_trip_cost_bp"],
    min_profit_to_cost_ratio=config["min_profit_to_cost_ratio"],
)

# Optional: load curves for "curve" mode
curves = None
build_pkg_fn = None
if config.get("mtm_mode") == "curve":
    print("Loading curves for full MTM mode...")
    import pytz
    tz = pytz.timezone("America/New_York")
    timestamps = [d.date() if hasattr(d, 'date') else d for d in rates_df.index]
    curve_map = curve_mdp.bulk_get_data({
        "curve_name": "USD-SOFR-1D",
        "timestamps": sorted(set(timestamps)),
        "n_jobs": 12,
    })
    curves = curve_map

    def _build_package(curve, fly_id, weights, direction):
        tenors = fly_id.split("/")
        pkg = []
        for j, t in enumerate(tenors):
            bpv = weights[j] * direction * (1 if j == 1 else -1)
            leg = curve.build_irswap(tenor=t, bpv=bpv * 1000)  # 1k/bp base
            pkg.append(leg)
        entry_npv = sum(curve.npv(leg) for leg in pkg)
        return pkg, entry_npv

    build_pkg_fn = _build_package
    print(f"Loaded {len(curve_map)} curves")

result = run_rv_backtest(
    residuals=all_residuals,
    zscores=all_zscores,
    rsq=all_rsq,
    weights=all_weights,
    rates=fly_dfs,
    regime=regime_series,
    fly_categories=fly_categories_map,
    config=bt_config,
    curves=curves,
    build_package_fn=build_pkg_fn,
)

print(f"Trades: {result.metrics['n_trades']}, Hit rate: {result.metrics['hit_rate']:.1%}, "
      f"Sharpe: {result.metrics['sharpe']:.2f}, Max DD: {result.metrics['max_drawdown_bp']:.1f}bp")

## Section 6: Analytics

In [ ]:
# Per-trade DataFrame
trade_df = pd.DataFrame([{
    "fly_id": t.fly_id,
    "category": t.category,
    "entry_date": t.entry_date,
    "exit_date": t.exit_date,
    "exit_reason": t.exit_reason,
    "entry_zscore": t.entry_zscore,
    "entry_residual_bp": t.entry_residual * 10_000,
    "pnl_bp": t.realized_pnl,
    "carry_bp": t.carry_bp,
    "holding_days": (len(pd.bdate_range(t.entry_date, t.exit_date)) - 1) if t.exit_date else 0,
    "direction": t.direction,
} for t in result.trades])
if len(trade_df) > 0:
    display(trade_df.describe())
    print("\nBy category:")
    display(trade_df.groupby("category")["pnl_bp"].agg(["count", "mean", "std", "sum"]))
    print("\nBy exit reason:")
    display(trade_df.groupby("exit_reason")["pnl_bp"].agg(["count", "mean", "sum"]))

In [ ]:
# Stationarity tests
stationarity_results = []
for fly_id, res in all_residuals.items():
    clean = res.dropna()
    if len(clean) > 50:
        adf = adf_test(clean)
        stationarity_results.append({
            "fly_id": fly_id,
            "adf_stat": adf["statistic"],
            "adf_pvalue": adf["pvalue"],
            "half_life_days": adf["half_life"],
            "stationary": adf["pvalue"] < 0.05,
        })
stationarity_df = pd.DataFrame(stationarity_results)
display(stationarity_df)

In [ ]:
# Parameter sensitivity grid (JPM Exhibit 7)
from itertools import product as itertools_product

rsq_thresholds = [0.60, 0.80]
residual_thresholds = [2.0, 3.0, 4.0]
zscore_thresholds = [1.5, 2.0]

sensitivity_results = []
for rsq_t, res_t, zs_t in itertools_product(rsq_thresholds, residual_thresholds, zscore_thresholds):
    test_config = RVBacktestConfig(
        mtm_mode="approximate",
        entry_min_rsq=rsq_t,
        entry_min_residual_bp=res_t,
        entry_min_zscore=zs_t,
        exit_mean_reversion=config["exit_mean_reversion"],
        exit_stop_loss_sd=config["exit_stop_loss_sd"],
        exit_max_holding_days=config["exit_max_holding_days"],
        round_trip_cost_bp=config["round_trip_cost_bp"],
    )
    r = run_rv_backtest(
        residuals=all_residuals, zscores=all_zscores, rsq=all_rsq,
        weights=all_weights, rates=fly_dfs, regime=regime_series,
        fly_categories=fly_categories_map, config=test_config,
    )
    sensitivity_results.append({
        "R²": rsq_t, "Residual_bp": res_t, "Z-score": zs_t,
        **r.metrics,
    })

sensitivity_df = pd.DataFrame(sensitivity_results)
display(sensitivity_df.style.format({
    "sharpe": "{:.2f}", "hit_rate": "{:.1%}", "avg_pnl_bp": "{:.2f}",
    "max_drawdown_bp": "{:.1f}", "total_pnl_bp": "{:.1f}",
}))

## Section 7: Visualization

In [ ]:
def _contiguous_regions(mask):
    """Yield (start, end) tuples for contiguous True regions in a boolean Series."""
    regions = []
    in_region = False
    start = None
    for dt, val in mask.items():
        if val and not in_region:
            start = dt
            in_region = True
        elif not val and in_region:
            regions.append((start, dt))
            in_region = False
    if in_region:
        regions.append((start, mask.index[-1]))
    return regions

In [ ]:
# Cumulative P&L + Drawdown
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 1]})

# Cumulative P&L by category
ax = axes[0]
result.cumulative_pnl.plot(ax=ax, label="Total", linewidth=2, color="black")
for cat in result.daily_pnl_by_category.columns:
    result.daily_pnl_by_category[cat].cumsum().plot(ax=ax, label=cat, alpha=0.7)
ax.set_title("Cumulative P&L (bp)")
ax.legend()
ax.grid(True, alpha=0.3)

# Drawdown
ax2 = axes[1]
dd = result.cumulative_pnl - result.cumulative_pnl.cummax()
dd.plot(ax=ax2, color="red", alpha=0.7)
ax2.fill_between(dd.index, dd.values, 0, alpha=0.3, color="red")
ax2.set_title("Drawdown (bp)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Quarterly P&L bar chart (JPM Exhibit 5)
quarterly_pnl = result.daily_pnl.resample("QE").sum()
colors = ["green" if x >= 0 else "red" for x in quarterly_pnl.values]
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(quarterly_pnl.index, quarterly_pnl.values, width=60, color=colors, alpha=0.7)
ax.set_title("Quarterly P&L (bp)")
ax.grid(True, alpha=0.3, axis="y")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Traffic light overlay
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(result.cumulative_pnl.index, result.cumulative_pnl.values, color="black", label="Cumulative P&L")
ax1.set_ylabel("P&L (bp)")

ax2 = ax1.twinx()
tl_indicator = tl_result["indicator"] if config["regime_filter_enabled"] else pd.Series()
if len(tl_indicator) > 0:
    ax2.plot(tl_indicator.index, tl_indicator.values, color="orange", alpha=0.5, label="Traffic Light")
    ax2.axhline(y=config["regime_threshold"], color="red", linestyle="--", alpha=0.5)
    # Shade red periods
    red_mask = tl_result["regime"] == "red"
    for start, end in _contiguous_regions(red_mask):
        ax1.axvspan(start, end, alpha=0.15, color="red")
    ax2.set_ylabel("Traffic Light Indicator")

ax1.legend(loc="upper left")
ax1.set_title("Cumulative P&L with Traffic Light Overlay")
plt.tight_layout()
plt.show()

In [ ]:
# Z-score heatmap (Standard Chartered Figure 12)
if config["pca_scope"] == "full_curve":
    # Run full-curve PCA for the heatmap
    full_pca_result = rolling_pca(rates_df.dropna(axis=1), pca_config)
    latest_zscores = full_pca_result.zscores.iloc[-1].dropna()

    fig, ax = plt.subplots(figsize=(12, 6))
    # Reshape into tenor x forward grid if possible
    ax.barh(range(len(latest_zscores)), latest_zscores.values)
    ax.set_yticks(range(len(latest_zscores)))
    ax.set_yticklabels(latest_zscores.index, fontsize=8)
    ax.axvline(x=0, color="black", linewidth=0.5)
    ax.set_title(f"PCA Residual Z-scores (latest: {full_pca_result.zscores.index[-1].strftime('%Y-%m-%d')})")
    ax.set_xlabel("Z-score (+ = cheap, - = rich)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-trade scatter
if len(trade_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].scatter(trade_df["entry_zscore"], trade_df["pnl_bp"], alpha=0.5, s=20)
    axes[0].axhline(y=0, color="black", linewidth=0.5)
    axes[0].set_xlabel("Entry Z-score")
    axes[0].set_ylabel("P&L (bp)")
    axes[0].set_title("P&L vs Entry Z-score")

    axes[1].scatter(trade_df["holding_days"], trade_df["pnl_bp"], alpha=0.5, s=20)
    axes[1].axhline(y=0, color="black", linewidth=0.5)
    axes[1].set_xlabel("Holding Period (days)")
    axes[1].set_ylabel("P&L (bp)")
    axes[1].set_title("P&L vs Holding Period")

    plt.tight_layout()
    plt.show()

In [ ]:
# Rolling beta charts
if config["regime_filter_enabled"] and ref_fly in fly_dfs:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    ref_reg.betas_body.dropna().plot(ax=axes[0], label="Beta vs Body", color="blue")
    axes[0].set_title(f"Rolling Regression Betas — {ref_fly}")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    ref_reg.betas_curve.dropna().plot(ax=axes[1], label="Beta vs Curve", color="green")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Shade red periods
    red_mask = tl_result["regime"] == "red"
    for ax in axes:
        for start, end in _contiguous_regions(red_mask):
            ax.axvspan(start, end, alpha=0.15, color="red")

    plt.tight_layout()
    plt.show()